In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
import sys, importlib, time
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
sys.path.insert(0, '/content/drive/MyDrive/RadarGestureEdgeAI')
import radar_utils as ru
importlib.reload(ru)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", DEVICE)
Xc, y, meta = ru.load_compact()
subj = meta['participant_id'].to_numpy()
print("Xc:", Xc.shape)

Device: cpu
Xc: (9593, 90, 32, 3)


In [ ]:
keep = np.flatnonzero(~meta['is_duplicate'].to_numpy())
rows = []
for seed in [42, 1042, 2042]:
    path = ru.PROJECT / 'results' / f'randsplit_M_s{seed}.csv'
    if path.exists():
        rows.append(pd.read_csv(path).iloc[0].to_dict()); print(f"seed {seed}: loaded"); continue
    t0 = time.time()
    trv, te = train_test_split(keep, test_size=0.2, stratify=y[keep], random_state=seed)
    tr, va = train_test_split(trv, test_size=0.1, stratify=y[trv], random_state=seed)
    fold = {'fold': 0, 'train_idx': np.sort(tr), 'val_idx': np.sort(va), 'test_idx': np.sort(te)}
    r = ru.train_cnn_fold(Xc, y, fold, DEVICE, seed=seed)
    m = ru.evaluate(y[te], r['y_pred'])
    row = {'seed': seed, 'accuracy': round(m['accuracy'] * 100, 2),
           'macro_f1': round(m['macro_f1'] * 100, 2), 'best_epoch': r['best_epoch']}
    pd.DataFrame([row]).to_csv(path, index=False); rows.append(row)
    print(f"seed {seed}: acc {row['accuracy']:.2f} | F1 {row['macro_f1']:.2f} | {(time.time()-t0)/60:.1f} min", flush=True)
rs = pd.DataFrame(rows)
print(f"\n===== RANDOM 80/20 SPLIT (model M, same person can be in train+test) =====")
print(rs.to_string(index=False))
print(f"Accuracy {rs.accuracy.mean():.2f} ± {rs.accuracy.std():.2f} | Macro-F1 {rs.macro_f1.mean():.2f} ± {rs.macro_f1.std():.2f}")
ru.save_results(rs, 'table_randsplit_M')

seed 42: acc 9.21 | F1 9.15 | 20.1 min
